In [ ]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

: 

In [0]:
dbutils.widgets.text("epic_secrets_scope", "", "Epic Secrets Scope")
dbutils.widgets.text("client_id_dbs_key", "", "Epic Client ID DB Secrets Key")
dbutils.widgets.text("token_url", "", "Epic Token URL")
dbutils.widgets.text("algo", "", "Epic Token Enrcyption Algorithm")

In [0]:
widget_values = dbutils.widgets.getAll()
for name, value in widget_values.items():
	print(f"{name} = {value}")

In [0]:
CLIENT_ID = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = widget_values["client_id_dbs_key"])
TOKEN_URL = widget_values["token_url"]
PRIVATE_KEY = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = "private_key")
ALGO = widget_values["algo"]
EPIC_KID = dbutils.secrets.get(scope=widget_values["epic_secrets_scope"], key="kid")

In [0]:
print(f"""
CLIENT_ID = {CLIENT_ID}
TOKEN_URL = {TOKEN_URL}
PRIVATE_KEY = {PRIVATE_KEY}
ALGO = {ALGO}
EPIC_KID = {EPIC_KID}
""")

In [0]:
import sys
import os

# Get the directory containing this notebook
notebook_dir = os.path.dirname(os.path.abspath('__file__'))

# Add src directory to path if not already present (notebook is already in src/)
if notebook_dir not in sys.path:
	sys.path.append(notebook_dir)

from epic_on_fhir.auth import EpicApiAuth
from epic_on_fhir.endpoint import EpicApiRequest

In [0]:
epicAuth = EpicApiAuth(
  client_id = CLIENT_ID,
  private_key = PRIVATE_KEY,
  kid = EPIC_KID,
  algo = ALGO,
  auth_location = TOKEN_URL
)

In [0]:
epicAuth.can_connect()

In [0]:
epicFhirAPi = EpicApiRequest(
    auth = epicAuth,
    base_url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/"
)

In [0]:
resource = "Patient"
action = "?identifier=EXTERNAL|Z6129"

In [0]:
patient = epicFhirAPi.make_request(
    http_method="get",
    resource=resource,
    action=action
)

In [0]:
import json

# Display the full response structure
print("Response keys:", patient.keys())
print("\nResponse status:", patient['response']['response_status_code'])
print("\nResponse text (first 500 chars):")
print(patient['response']['response_text'][:500])

In [0]:
# Parse the response text as JSON
patient_data = json.loads(patient['response']['response_text'])

# Display the parsed patient data
display(patient_data)

In [0]:
# Extract the FHIR identifier from the patient resource
identifiers = patient_data['entry'][0]['resource']['identifier']
patient_id = None

for identifier in identifiers:
	if identifier.get('type', {}).get('text') == 'FHIR':
		patient_id = identifier['value']
		break

print(f"Patient FHIR ID: {patient_id}")

In [0]:
patient_id = "erXuFYUfucBZaryVksYEcMg3"

In [0]:
resource = "Patient"
action = f"{patient_id}/$summary"
print(action)

In [0]:
epicFhirAPi.make_request(
    http_method="get",
    resource=resource,
    action=action
)